# 00 · Inspect the raw data

Quick look at `data/raw/balanced-reviews.txt` before building the pipeline: figure out the file's encoding, delimiter, columns and types, so the bronze step (`src/01_bronze.ipynb`) can read it with an explicit schema.

## Raw bytes

Read the first 200 bytes in binary mode to see the encoding and delimiter without guessing.

In [ ]:
with open("../data/raw/balanced-reviews.txt", "rb") as f:
    print(f.read(200))

**Findings from the raw bytes:**

- Starts with `\xff\xfe` → a byte-order mark for **UTF-16 (little-endian)**, so it must be read with `encoding="utf-16"`, not the default UTF-8.
- Fields are separated by `\t` → **tab-separated**, with a header row.
- Lines end with `\r\n` (Windows line endings).

## Load with pandas

Load the full file to see the columns, types and row count.

In [8]:
import pandas as pd

df = pd.read_csv("../data/raw/balanced-reviews.txt",
                  sep="\t", encoding="UTF_16")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 105698 entries, 0 to 105697
Data columns (total 7 columns):
 #   Column      Non-Null Count   Dtype
---  ------      --------------   -----
 0   no          105698 non-null  int64
 1   Hotel name  105698 non-null  str  
 2   rating      105698 non-null  int64
 3   user type   105698 non-null  str  
 4   room type   105698 non-null  str  
 5   nights      105698 non-null  str  
 6   review      105698 non-null  str  
dtypes: int64(2), str(5)
memory usage: 40.7 MB


**Summary:**

- **105,698 reviews**, 7 columns, no nulls.
- `no` and `rating` are integers; everything else is text.
- `nights` is free text in Arabic (e.g. `أقمت ليلة واحدة`, "I stayed one night") and needs parsing later.
- `room type` uses `-` as a placeholder for missing values — it shows as non-null here but is really missing.

These findings drive the explicit schema in `src/01_bronze.ipynb`.